In [ ]:




#------------------------------------------------ Begin_Librairie ----------------------------------------



import pandas as pd 



from time import sleep



import datetime



from pandas import ExcelWriter



import re



import pdfplumber



import os



from bs4 import BeautifulSoup



from selenium import webdriver



from selenium.webdriver.common.by import By



from selenium.webdriver.common.keys import Keys



from webdriver_manager.chrome import ChromeDriverManager



from selenium.webdriver.chrome.options import Options



from selenium.webdriver.common.alert import Alert



from googletrans import Translator

import sys
sys.stdout.reconfigure(encoding='utf-8')

# %%



#------------------------------------------------ Begin_ fileName ----------------------------------------



regulatorName = 'CN CSRC'



print(f"Running {regulatorName} Web Scraping Tool v.1.1")



now=datetime.datetime.now()



filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])



#scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}" ## to comment for the local environment



scriptfolder=os.path.dirname(os.path.abspath(__file__))



os.chdir(scriptfolder)



writer = ExcelWriter(filename)



tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process







if os.path.exists(tempfolder):



    for rem in os.listdir(tempfolder):



        os.remove(os.path.join(tempfolder, rem))



else:



    os.mkdir(tempfolder)



# %%



#------------------------------------------------ Begin_chromedriver ----------------------------------------







#Starting Chrome driver, set to download files in tempfolder

#Try to download the insecure file in 







chrome_options = Options()

chrome_options.add_argument("--window-size=1920,1080")

chrome_options.add_argument("--allow-running-insecure-content")  # Allow insecure content



chrome_options.add_experimental_option("prefs", {

    "download.default_directory": tempfolder,

    "download.prompt_for_download": False,

    "download.directory_upgrade": True,

    "safebrowsing.enabled": True

})



driver = webdriver.Chrome(options=chrome_options)

driver.maximize_window()

#------------------------------------------------ Begin_Fouction ----------------------------------------





def bourange_same_length_array(sqldict) :



    maxlen = len(sqldict['ListProcessDate'])



    for key, val in sqldict.items():



        if len(sqldict[key]) != maxlen:



            empty = []



            total_empty = maxlen - len(sqldict[key])



            for i in range(total_empty):



                empty.append('')



            sqldict[key]=sqldict[key]+empty



    return sqldict





# Initialize the translator

translator = Translator()

def translate_text(text):
    
    sleep(1)

    return translator.translate(text, src='zh-cn', dest='en').text

    

# %%



#------------------------------------------------ Begin_Variable ----------------------------------------



regdict = { 'CN CSRC 1': 'http://www.csrc.gov.cn/',

            'CN CSRC 2': 'http://www.csrc.gov.cn/',

            'CN CSRC 3': 'http://www.csrc.gov.cn/',

            'CN CSRC 4': 'http://www.csrc.gov.cn/',

            'CN CSRC 5': 'http://www.csrc.gov.cn/',

            }



Typology ={



            'CN CSRC 1': 'List of Securities Companies',

            'CN CSRC 2': 'List of Futures Companies',

            'CN CSRC 3': 'List of Fund Management Companies',

            'CN CSRC 4': 'List of QFIIs',

            'CN CSRC 5': 'List of Custodian Banks for Qualified Foreign Investors',



}



sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'Name_2':[],'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}



processdate = now.strftime('%Y-%m-%d')



# %%

#------------------------------------------------ Begin_Main ----------------------------------------
for reg in regdict:
    
    driver = webdriver.Chrome(options=chrome_options)
    driver.maximize_window()
    sleep(2)
    print(f'Working with list {reg}')
    driver.get(regdict[reg])
    
    input_element = driver.find_element(By.ID, 'searchWord')
    # Enter the search term
    if reg == 'CN CSRC 1' :
        input_element.send_keys('证券公司')
        print('Input Chinese Keywords ')
        # Submit the form
        input_element.send_keys(Keys.RETURN)
        sleep(3)
        window_handles = driver.window_handles
        if len(window_handles)>1:
            driver.switch_to.window(window_handles[1])
        else:
            print('Can not open the second Tab')    
        sleep(2)
        soup = BeautifulSoup(driver.page_source, 'html.parser')  
        newsInfo = soup.find('div',{'class':'newsInfo'}) 
        innerlinks = newsInfo.find('ul').find_all('a')
        for inner in innerlinks:
            if '名录' in inner['data'] :
                print(inner['href'])
                chrome_options.add_argument("--unsafely-treat-insecure-origin-as-secure="+inner['href'])  # Replace example.com with your site's domain
                driver.quit()
                sleep(2)
                driver2 = webdriver.Chrome(options=chrome_options)
                driver2.get(inner['href'])
                sleep(3)

        soup2 = BeautifulSoup(driver2.page_source, 'html.parser')  
        driver2.find_element(By.XPATH,'//*[@id="files"]').click()
        print('Download the file')
        sleep(5)
        driver2.quit()
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        sleep(2)
        dataframe = pd.read_excel(dl_files[0], engine='xlrd')
        dataframe = dataframe.iloc[:,-2:]

        # Apply the translation function to each cell in the dataframe
        # 5 min about 150 data of two columns
        translated_dataframe = pd.DataFrame()
        translated_dataframe['CompanyName'] = dataframe['公司名称'].map(translate_text)
        translated_dataframe['RegistedCity'] =  dataframe['辖区（注册地）'].map(translate_text)
        translated_dataframe['CompanyNameCN'] = dataframe['公司名称']
        #translated_dataframe = dataframe.map(translate_text)
        #translated_dataframe =  translated_dataframe.rename(columns={"公司名称": "CompanyName", "辖区（注册地）": "Registed City"})
        for name,city,cname in zip(translated_dataframe['CompanyName'],translated_dataframe['RegistedCity'],translated_dataframe['CompanyNameCN']):
            sqldict['Name'].append(cname)
            sqldict['Name_2'].append(name)
            sqldict['City'].append(city)
            sqldict['ListProcessDate'].append(processdate)    
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
        sqldict = bourange_same_length_array(sqldict)
        
    elif reg == 'CN CSRC 3':
        input_element.send_keys('基金管理机构')
        print('Input Chinese Keywords ')
        # Submit the form
        input_element.send_keys(Keys.RETURN)
        sleep(3)
        window_handles = driver.window_handles
        if len(window_handles)>1:
            driver.switch_to.window(window_handles[1])
        else:
            print('Can not open the second Tab')    
        # sleep(2)
        # soup = BeautifulSoup(driver.page_source, 'html.parser')  
        # newsInfo = soup.find('div',{'class':'wordGuide Residence-permit'}) 
        sleep(2)
        soup = BeautifulSoup(driver.page_source, 'html.parser')  
        newsInfo = soup.find('div',{'class':'newsInfo'}) 
        innerlinks = newsInfo.find('ul').find_all('a')
        for inner in innerlinks:
            if '名录' in inner['data'] :
                print(inner['href'])
                chrome_options.add_argument("--unsafely-treat-insecure-origin-as-secure="+inner['href'])  # Replace example.com with your site's domain
                driver.quit()
                sleep(2)
                driver2 = webdriver.Chrome(options=chrome_options)
                driver2.get(inner['href'])
                sleep(3)

        soup2 = BeautifulSoup(driver2.page_source, 'html.parser')  
        driver2.find_element(By.XPATH,'//*[@id="files"]').click()
        print('Download the file')
        sleep(2)
        driver2.quit()
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        sleep(2)
        dataframe = pd.read_excel(dl_files[0], engine='xlrd')
        dataframe = dataframe.iloc[:,-2:]

        # Apply the translation function to each cell in the dataframe
        # 5 min about 150 data of two columns
        translated_dataframe = pd.DataFrame()
        translated_dataframe['CompanyName'] = dataframe['公司名称'].map(translate_text)
        translated_dataframe['RegistedCity'] =  dataframe['辖区（注册地）'].map(translate_text)
        translated_dataframe['CompanyNameCN'] = dataframe['公司名称']
        #translated_dataframe = dataframe.map(translate_text)
        #translated_dataframe =  translated_dataframe.rename(columns={"公司名称": "CompanyName", "辖区（注册地）": "Registed City"})
        for name,city,cname in zip(translated_dataframe['CompanyName'],translated_dataframe['RegistedCity'],translated_dataframe['CompanyNameCN']):
            sqldict['Name'].append(cname)
            sqldict['Name_2'].append(name)
            sqldict['City'].append(city)
            sqldict['ListProcessDate'].append(processdate)    
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
        sqldict = bourange_same_length_array(sqldict)
        
    
    elif  reg == 'CN CSRC 2': 
        input_element.send_keys('期货公司')
        print('Input Chinese Keywords ')
        # Submit the form
        input_element.send_keys(Keys.RETURN)
        sleep(3)
        window_handles = driver.window_handles
        if len(window_handles)>1:
            driver.switch_to.window(window_handles[1])
        else:
            print('Can not open the second Tab')    
        sleep(2)
        soup = BeautifulSoup(driver.page_source, 'html.parser')  
        newsInfo = soup.find('div',{'class':'newsInfo'}) 
        innerlinks = newsInfo.find('ul').find_all('a')
        for inner in innerlinks:
            if '名录' in inner['data'] :
                print(inner['href'])
                chrome_options.add_argument("--unsafely-treat-insecure-origin-as-secure="+inner['href'])  # Replace example.com with your site's domain
                driver.quit()
                sleep(2)
                driver2 = webdriver.Chrome(options=chrome_options)
                driver2.get(inner['href'])
                sleep(3)

        print('Try to Search Documents in the news information list')
        soup2 = BeautifulSoup(driver2.page_source, 'html.parser')  
        driver2.find_element(By.XPATH,'//*[@id="files"]').click()
        print('Download the file')
        sleep(2)
        driver2.quit()
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        sleep(2)
        dataframe = pd.read_excel(dl_files[0], engine='xlrd')
        dataframe = dataframe.iloc[:,-2:]
        dataframe.replace('NaN', pd.NA, inplace=True)
        # Forward fill the NaN values
        dataframe['辖区'].fillna(method='ffill', inplace=True)
        #print(dataframe)
        # Apply the translation function to each cell in the dataframe
        # 5 min about 150 data of two columns
        translated_dataframe = pd.DataFrame()
        translated_dataframe['CompanyNameCN'] = dataframe['期货公司名称']
        translated_dataframe['CompanyName'] = dataframe['期货公司名称'].map(translate_text)
        translated_dataframe['RegistedCity'] =  dataframe['辖区'].map(translate_text)
        #translated_dataframe = dataframe.map(translate_text)
        #translated_dataframe =  translated_dataframe.rename(columns={"辖区": "Registed City","期货公司名称": "CompanyName", })
        for name,city,cname in zip(translated_dataframe['CompanyName'],translated_dataframe['RegistedCity'],translated_dataframe['CompanyNameCN']):
            sqldict['Name'].append(cname)
            sqldict['Name_2'].append(name)
            sqldict['City'].append(city)
            sqldict['ListProcessDate'].append(processdate)    
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
        sqldict = bourange_same_length_array(sqldict)


    elif reg == 'CN CSRC 4' :
        
        input_element.send_keys('合格境外投资者')
        print('Input Chinese Keywords ')
        # Submit the form
        input_element.send_keys(Keys.RETURN)
        sleep(3)
        window_handles = driver.window_handles
        if len(window_handles)>1:
            driver.switch_to.window(window_handles[1])
        else:
            print('Can not open the second Tab')    
        sleep(2)
        soup = BeautifulSoup(driver.page_source, 'html.parser')  
        newsInfo = soup.find('div',{'class':'newsInfo'}) 
        innerlinks = newsInfo.find('ul').find_all('a')
        inner = innerlinks[0]
        if '名录' in inner['data'] :
            print(inner['href'])
            chrome_options.add_argument("--unsafely-treat-insecure-origin-as-secure="+inner['href'])  # Replace example.com with your site's domain
            driver.quit()
            sleep(2)
            driver2 = webdriver.Chrome(options=chrome_options)
            driver2.get(inner['href'])
            sleep(3)


        soup2 = BeautifulSoup(driver2.page_source, 'html.parser')  
        driver2.find_element(By.XPATH,'//*[@id="files"]').click()
        print('Download the file')
        sleep(2)
        driver2.quit()
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        sleep(2)
        dataframe = pd.read_excel(dl_files[0], engine='xlrd')
        dataframe.columns = dataframe.iloc[0]
        dataframe = dataframe[1:].iloc[:,1:]
        dataframe['英文名称'].fillna('', inplace=True)
        for ENname,CNname,city,ApproveDate in zip(dataframe['英文名称'],dataframe['中文名称'],dataframe['注册地'],dataframe['批准日期']):
            if CNname!='':
                sqldict['ListProcessDate'].append(processdate)    
                sqldict['RegulationType'].append('Regulated')
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['ListName'].append(Typology[reg])
                translate_cntry = translate_text(city)
                sqldict['Cntry'].append(translate_cntry)
                sqldict['RegulationDate'].append(ApproveDate)
                if ENname=='':
                    print(ENname,CNname,city,ApproveDate)
                    translate_name = translate_text(CNname)
                    sqldict['Name'].append(CNname)
                    sqldict['Name_2'].append(translate_name)
                    #print(translate_name)
                else:
                    sqldict['Name'].append(CNname)
                    sqldict['Name_2'].append(ENname)
                
        sqldict = bourange_same_length_array(sqldict)


    elif reg == 'CN CSRC 5':

        input_element.send_keys('合格境外投资者托管行')
        print('Input Chinese Keywords ')
        # Submit the form
        input_element.send_keys(Keys.RETURN)
        sleep(3)
        window_handles = driver.window_handles
        if len(window_handles)>1:
            driver.switch_to.window(window_handles[1])
        else:
            print('Can not open the second Tab')    
        sleep(2)
        try:
            soup = BeautifulSoup(driver.page_source, 'html.parser')  
            sleep(2)
            wordguide = soup.find('div',{'class':'wordGuide Residence-permit'} )
            sleep(2)
            innerlink = wordguide.find('a')['href']
        except:
            driver.refresh()
            sleep(2)
            soup = BeautifulSoup(driver.page_source, 'html.parser')  
            sleep(2)
            wordguide = soup.find('div',{'class':'wordGuide Residence-permit'} )
            sleep(2)
            innerlink = wordguide.find('a')['href']
            
        chrome_options.add_argument("--unsafely-treat-insecure-origin-as-secure="+innerlink)  # Replace example.com with your site's domain
        driver.quit()
        sleep(2)
        driver2 = webdriver.Chrome(options=chrome_options)
        driver2.get(innerlink)
        sleep(3)
        print('Try to Search Documents in the news information list')
        soup2 = BeautifulSoup(driver2.page_source, 'html.parser')  
        driver2.find_element(By.XPATH,'//*[@id="files"]').click()
        print('Download the file')
        sleep(2)
        driver2.quit()
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        sleep(2)
        dataframe = pd.read_excel(dl_files[0], engine='xlrd')
        for info,cname in zip(dataframe['合格境外投资者托管行英文名称'],dataframe['合格境外投资者托管行中文名称']):
            print(info)
            sqldict['Name'].append(cname)
            sqldict['Name_2'].append(info)
            sqldict['ListProcessDate'].append(processdate)    
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
        sqldict = bourange_same_length_array(sqldict)
    
                
                        
    
    if os.path.exists(tempfolder):

        for rem in os.listdir(tempfolder):

            os.remove(os.path.join(tempfolder, rem))
# %%



#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------



os.chdir(scriptfolder)



df=pd.DataFrame(sqldict)



df = df.drop_duplicates()



df.to_excel(writer, 'SQL Ready', index=False)



writer.save()



writer.close()



driver.quit()



sleep(3)

    




    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    